# AkasicDB 하이브리드 검색 데모 — Qwen3-Embedding-0.6B (Colab)

합성 상품 데이터를 만들고, 설명문을 **Qwen3-Embedding-0.6B**(1024차원)로 인코딩해 AkasicDB에 적재한 뒤,
**그래프(Cypher) + 키워드(BM25) + 벡터(L2) + SQL 필터**를 한 쿼리로 실행하고 실행 계획까지 확인합니다.

**준비물**
1. [db.akasic.cloud](https://db.akasic.cloud)에서 인스턴스 생성 후, 연결 문자열을 Colab Secrets에 `DB_DSN`으로 저장 (좌측 🔑 아이콘)
2. 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택

> 기본 데이터 규모는 10,000행입니다. T4 기준 인코딩에 수 분 걸립니다. 100,000행으로 늘리면 30분 이상 걸리니 필요할 때만 조절하세요.

In [ ]:
%pip install --quiet akasicdb "psycopg[binary]" "sentence-transformers>=2.7" "transformers>=4.51" accelerate

## 1. DB 연결

In [ ]:
import os, psycopg
from google.colab import userdata
from akasicdb.psycopg import register_vector

DB_DSN = os.getenv("DATABASE_URL") or userdata.get("DB_DSN")
conn = psycopg.connect(DB_DSN, autocommit=True)
register_vector(conn)

print(conn.execute("SELECT version()").fetchone()[0])

## 2. 스키마 생성\n\nQwen3-Embedding-0.6B의 출력 차원에 맞춰 `vectoron.vector(1024)`를 사용합니다.

In [ ]:
conn.execute("DROP TABLE IF EXISTS products, brands CASCADE")

conn.execute("""
CREATE TABLE brands (
  b_id  integer PRIMARY KEY,
  name  text NOT NULL,
  hot   boolean NOT NULL DEFAULT false
)""")

conn.execute("""
CREATE TABLE products (
  id       integer PRIMARY KEY,
  b_id     integer NOT NULL REFERENCES brands(b_id),
  title    text NOT NULL,
  category text NOT NULL,
  price    integer NOT NULL,
  descr    text NOT NULL,
  emb      vectoron.vector(1024)
)""")
print("ok")

## 3. 합성 데이터 생성 (Python)\n\n`descr`는 어휘 풀 조합이라 `'여름 AND 러닝화'` BM25 질의에 일부 행만 매칭됩니다.

In [ ]:
import random
random.seed(42)

N_BRANDS, N_PRODUCTS = 50, 10_000

seasons = ['여름', '겨울', '사계절', '봄가을']
items   = ['러닝화', '운동화', '샌들', '트레킹화', '슬립온']
feats   = ['통기성이 좋은', '가볍고 시원한', '쿠션이 좋은', '방수가 되는']
cats    = ['shoes', 'apparel', 'bag', 'acc', 'outdoor']

brands = [(i, f'brand_{i}', i % 5 == 0) for i in range(1, N_BRANDS + 1)]

products = []
for g in range(1, N_PRODUCTS + 1):
    descr = f"{seasons[(g // 7) % 4]} 시즌 {items[(g // 3) % 5]} — {feats[g % 4]} 제품입니다."
    products.append((
        g,                              # id
        (g % N_BRANDS) + 1,             # b_id
        f'product_{g}',                 # title
        cats[g % 5],                    # category
        10_000 + random.randrange(190_000),  # price
        descr,
    ))

print(products[0])
print(products[1])

## 4. Qwen3-Embedding-0.6B 로드 및 인코딩\n\n문서는 instruction 없이, 검색 질의는 `prompt_name=\"query\"`로 인코딩하는 것이 이 모델의 사용법입니다. `normalize_embeddings=True`로 정규화하면 `<->`(L2) 정렬이 코사인 정렬과 동일해집니다.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"torch_dtype": torch.float16},
    tokenizer_kwargs={"padding_side": "left"},
)

descrs = [p[5] for p in products]
embs = model.encode(
    descrs,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(embs.shape)  # (10000, 1024)

## 5. 적재

In [ ]:
with conn.cursor() as cur:
    cur.executemany("INSERT INTO brands VALUES (%s, %s, %s)", brands)

    rows = [(*p, emb) for p, emb in zip(products, embs)]
    BATCH = 1000
    for i in range(0, len(rows), BATCH):
        cur.executemany(
            "INSERT INTO products (id, b_id, title, category, price, descr, emb) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            rows[i:i + BATCH],
        )
        print(f"{min(i + BATCH, len(rows))} / {len(rows)}")

conn.execute("ANALYZE brands, products")
print("done")

## 6. 인덱스 3종 — 벡터(vectoron) · 전문검색(texton) · B-tree

In [ ]:
conn.execute("CREATE INDEX products_emb_idx   ON products USING vectoron (emb vector_l2_ops)")
conn.execute("CREATE INDEX products_descr_idx ON products USING texton (descr)")
conn.execute("CREATE INDEX products_cat_idx   ON products (category)")
print("ok")

## 7. 그래프 정의 — 기존 테이블에서 정점·간선 파생 (재적재 없음)

In [ ]:
conn.execute("SELECT akasicdb.define_graph('retail_graph')")

conn.execute("""SELECT akasicdb.define_vertex('retail_graph', 'v_product',
  ARRAY['product_id integer', 'title text'],
  'products',
  'SELECT id AS product_id, title FROM products')""")

conn.execute("""SELECT akasicdb.define_vertex('retail_graph', 'v_brand',
  ARRAY['b_id integer', 'name text', 'hot boolean'],
  'brands',
  'SELECT b_id, name, hot FROM brands')""")

conn.execute("""SELECT akasicdb.define_edge('retail_graph', 'e_same_line',
  'v_product', 'v_brand',
  null,
  'SELECT null FROM products p, brands b WHERE p.b_id = b.b_id',
  'products p', 'brands b')""")

conn.execute("SELECT akasicdb.create_graph('retail_graph')")
print("graph ready")

## 8. 하이브리드 질의 실행\n\n검색 질의문은 `prompt_name=\"query\"`로 인코딩합니다.

In [ ]:
q_emb = model.encode(
    ["여름에 신을 시원한 러닝화"],
    prompt_name="query",
    normalize_embeddings=True,
)[0]

HYBRID_SQL = """
SELECT p.id, p.title, p.price, left(p.descr, 40) AS descr
FROM   products p
JOIN   akasicdb.cypher('retail_graph', $$
           MATCH (pr:v_product)-[:e_same_line]->(b:v_brand)
           WHERE b.hot = true
           RETURN pr.product_id AS product_id
       $$) AS g (product_id integer)
       ON g.product_id = p.id                          -- 그래프 탐색 (Cypher)
WHERE  p.category = 'shoes'                            -- SQL 필터
  AND  p.descr @@ texton.query('여름 AND 러닝화')        -- 키워드 (BM25)
ORDER  BY p.emb <-> %s::vectoron.vector                -- 벡터 유사도 (L2)
LIMIT  10
"""

for row in conn.execute(HYBRID_SQL, (q_emb,)):
    print(row)

## 9. 실행 계획 확인 — 표준 PostgreSQL EXPLAIN 그대로

In [ ]:
plan = conn.execute("EXPLAIN (ANALYZE, BUFFERS) " + HYBRID_SQL, (q_emb,)).fetchall()
for (line,) in plan:
    print(line)

### 계획에서 읽을 포인트

1. **texton 조건** — `Index Scan using products_descr_idx`(또는 Bitmap Index Scan)로 나타나야 FTS 인덱스를 탄 것입니다. `Seq Scan` + `Filter`로 나오면 인덱스를 못 탄 것.
2. **벡터 정렬** — `Index Scan using products_emb_idx`면 ANN 인덱스 경로, 별도 `Sort` 노드가 보이면 정확 검색(전수 계산)입니다. 이 예제처럼 선택도 높은 필터(BM25 + category + 그래프 조인)가 함께 있으면 플래너가 의도적으로 후자를 고를 수 있고, 후보가 수백 건 수준이면 그쪽이 오히려 빠릅니다.
3. **cypher()** — 외부 계획에는 `Function Scan on cypher g` 한 줄로 보입니다. 그래프 내부 실행이 궁금하면 `SELECT * FROM akasicdb.cypher(...)`를 단독으로 EXPLAIN 해보세요.

필터 조건(`WHERE b.hot`, `category`, BM25 질의어)을 바꿔가며 플래너가 경로를 어떻게 바꾸는지 비교해 보는 것도 좋은 데모가 됩니다.